# LIBERO Phase 1: ID, OOD и early failure signals

Дата среза: **24 июля 2026 года**.

Этот notebook содержит только результаты серверной Phase 1. Полный протокол
находится в [LIBERO_OOD_SAFETY_CAMPAIGN.md](LIBERO_OOD_SAFETY_CAMPAIGN.md),
машинные таблицы и полный отчёт — в
[`campaigns/phase1_analysis_20260724`](campaigns/phase1_analysis_20260724/README.md).

В анализ включены только завершённые non-smoke runs с `query_traces`,
`metadata` и `pair_summary`. Оборванный запуск с отсутствующими assets исключён.

In [ ]:
from pathlib import Path
import json
import html
import pandas as pd
from IPython.display import HTML, Image, Markdown, Video, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'experiments':
    PROJECT_ROOT = PROJECT_ROOT.parent

ANALYSIS_DIR = PROJECT_ROOT / 'experiments/campaigns/phase1_analysis_20260724'
summary = json.loads((ANALYSIS_DIR / 'summary.json').read_text(encoding='utf-8'))
outcomes = pd.read_csv(ANALYSIS_DIR / 'outcomes_by_suite_task.csv')
episodes = pd.read_csv(ANALYSIS_DIR / 'episode_outcomes.csv')
ranking = pd.read_csv(ANALYSIS_DIR / 'early_online_metric_ranking_q0_3.csv')
metric_by_group = pd.read_csv(ANALYSIS_DIR / 'early_online_metric_by_group_q0_3.csv')
correlations = pd.read_csv(
    ANALYSIS_DIR / 'uncertainty_prediction_error_correlations_controlled.csv'
)
videos = pd.read_csv(ANALYSIS_DIR / 'video_inventory.csv')
print('Loaded:', len(episodes), 'episodes')

## 1. Task outcomes

| Split | Эпизоды | Success | Fail | Success rate | Wilson 95% CI |
|---|---:|---:|---:|---:|---:|
| ID | 72 | 72 | 0 | 100.0% | [94.9%, 100.0%] |
| LIBERO-PRO OOD | 106 | 82 | 24 | 77.4% | [68.5%, 84.3%] |

Все 72 ID rollout завершились успешно. На специально отобранных OOD
конфигурациях success rate снизился до 77.4%. Это подтверждает работоспособность
OOD screening, но не является несмещённой оценкой полного LIBERO-PRO benchmark.

In [ ]:
display(outcomes)
display(Image(filename=str(ANALYSIS_DIR / 'plots/success_rate_by_suite_task.png')))

## 2. Fixed-init mixed success/fail

| Suite | Task | Init | Success | Success rate |
|---|---:|---:|---:|---:|
| `libero_10_with_milk` | 9 | 0 | 3/4 | 75.0% |
| `libero_10_with_mug` | 4 | 0 | 2/4 | 50.0% |
| `libero_goal_with_mug` | 9 | 0 | 3/4 | 75.0% |
| `libero_spatial_with_milk` | 5 | 0 | 12/24 | 50.0% |
| `libero_spatial_with_mug` | 0 | 0 | 8/12 | 66.7% |
| `libero_spatial_with_yellow_book` | 5 | 0 | 5/6 | 83.3% |
| `libero_spatial_with_yellow_book` | 8 | 0 | 3/6 | 50.0% |

Получено **7** конфигураций, где при одинаковых
`suite/task/init_state` разные rollout seeds дали оба исхода. Самые удобные
границы для planning:

- `libero_spatial_with_milk/task5/init0`: 12/24 success;
- `libero_spatial_with_yellow_book/task8/init0`: 3/6;
- `libero_10_with_mug/task4/init0`: 2/4.

Milk уже даёт наиболее надёжную статистику. Yellow-book и long-horizon варианты
нужно расширить минимум до 40 rollout каждый.

In [ ]:
mixed = outcomes[
    outcomes['success'].gt(0) & outcomes['fail'].gt(0)
].copy()
display(mixed.sort_values(['success_rate', 'episodes']))

## 3. Failure modes и safety diagnostics

OOD outcomes:

- `timeout_no_goal`: **12**;
- `target_drop_candidate`: **11**;
- `timeout_partial_goal`: **1**.

Официальных safety violations нет, поскольку Phase 1 запускалась в LIBERO-PRO,
а не в LIBERO-Safety. Drop heuristic отметил 19 OOD эпизодов, но только 11 из
них закончились fail: precision 57.9%, recall всех fail 45.8%. Восемь
траекторий восстановились и завершили задачу, поэтому transient drop полезен как
событие для temporal monitor, но не как окончательный label.

In [ ]:
display(Image(filename=str(ANALYSIS_DIR / 'plots/ood_failure_modes.png')))
display(
    pd.crosstab(
        episodes.loc[episodes['split'].eq('OOD'), 'success'],
        episodes.loc[episodes['split'].eq('OOD'), 'target_drop_candidate'],
        margins=True,
    )
)

## 4. Early online failure signal

Для каждого эпизода online-метрики усредняются только по `query=0..3`
(`t=0..48`) и затем z-нормализуются внутри того же `suite/task/init_state`.
Самое раннее зарегистрированное failure event произошло на `t=55`, поэтому
анализ не использует post-failure данные.

Для action latent модель несколько раз кодирует одну и ту же action sequence.
Использованный internal-consistency signal:

$$
U_{copy}^A =
\frac{1}{N}
\sum_{i=1}^N
\operatorname{mean}_{t,d}
\operatorname{std}_k
L^A_{i,k,t,d}.
$$

Здесь $i$ — stochastic sample, $k$ — повтор action внутри latent frame.

| Early metric, mean over query 0-3 | Pooled within-group AUROC | Direction | High / tie / low groups |
|---|---:|---|---:|
| `latent_action_copy_std_mean_mean_over_samples` | 0.671 | `high=failure` | 5 / 1 / 1 |
| `value_mean` | 0.594 | `high=failure` | 3 / 1 / 3 |
| `future_image_pixel_std_mean` | 0.579 | `high=failure` | 4 / 0 / 3 |
| `action_std_mean` | 0.576 | `high=failure` | 3 / 0 / 4 |
| `value_std` | 0.537 | `high=failure` | 4 / 1 / 2 |
| `value_range` | 0.522 | `high=failure` | 4 / 1 / 2 |
| `action_first_step_l2_std` | 0.512 | `high=failure` | 2 / 0 / 5 |

Лучший exploratory результат:
`latent_action_copy_std_mean_mean_over_samples`, AUROC **0.671**. Направление
`high=failure` выполняется в 5/7 групп, одна группа даёт tie и одна инверсию.
Это кандидат для holdout, а не готовый универсальный detector.

In [ ]:
display(ranking.head(15))
display(Image(filename=str(ANALYSIS_DIR / 'plots/early_online_metric_auc.png')))
display(Image(filename=str(ANALYSIS_DIR / 'plots/top_early_metric_by_group.png')))

## 5. Prediction error after executing a chunk

Raw pooled correlations были высокими, потому что predicted value, uncertainty
и prediction error одновременно меняются по фазе эпизода. После
z-нормализации внутри каждого `suite/task/init_state/query_idx`:

- internal future-proprio consistency против future proprio L2:
  `rho = 0.139`;
- latent action consistency против future image MSE: связь ещё слабее;
- высокая связь `value_mean` с value-vs-chunk-success error частично
  тавтологична, поскольку target почти всегда равен нулю до terminal chunk.

Следовательно, Phase 1 не доказывает, что stochastic dispersion хорошо
калибрует ошибку world model.

In [ ]:
non_value = correlations[
    correlations['prediction_error_metric'].ne(
        'prediction_error_value_abs_chunk_success'
    )
]
display(non_value.head(20))

## 6. Milk boundary videos: 5 success и 5 fail

Видео записаны полностью: success до момента выполнения цели, fail до
`t=220`. Они нужны для проверки того, что timeout действительно соответствует
неудачной манипуляции, а drop detector не путает восстановившиеся эпизоды с
окончательным fail.

In [ ]:
available = videos[videos['local_video_path'].fillna('').ne('')].copy()
success_videos = available[available['success'].astype(bool)].head(5)
failed_videos = available[~available['success'].astype(bool)].head(5)

rows = []
for index in range(max(len(success_videos), len(failed_videos))):
    cells = []
    for label, frame in [('SUCCESS', success_videos), ('FAIL', failed_videos)]:
        if index >= len(frame):
            cells.append('<td></td>')
            continue
        row = frame.iloc[index]
        path = PROJECT_ROOT / row['local_video_path']
        video = Video(
            str(path),
            embed=True,
            html_attributes='controls preload="metadata" width="460"',
        )._repr_html_()
        caption = (
            f"{label}: seed={int(row['rollout_seed'])}, "
            f"rollout={int(row['rollout_id'])}"
        )
        cells.append(
            '<td style="vertical-align:top;padding:8px">'
            f'<b>{html.escape(caption)}</b><br>{video}</td>'
        )
    rows.append('<tr>' + ''.join(cells) + '</tr>')
display(HTML('<table>' + ''.join(rows) + '</table>'))

## 7. Главные выводы и следующий тест

1. OOD suites действительно создают natural fail при сохранении значимого
   числа success.
2. Internal action-latent consistency переносится лучше остальных ранних
   признаков, но AUROC 0.671 недостаточен для автономного решения.
3. Ранее сильные `action_first_step_std` и value-overconfidence выводы были
   специфичны для одной milk-конфигурации.
4. Output uncertainty почти не объясняет future prediction error после
   контроля фазы эпизода.
5. Drop events следует моделировать как temporal события с возможным recovery.

Следующий честный эксперимент: calibration на одном seed block, затем
зафиксированный predictor на новых seeds и новых init states. После этого
проверяется paired planning `max(value)` против
`value - lambda * calibrated_risk` на одинаковых candidate sets.